In [7]:
import pyspark
from pyspark import SparkContext

conf: pyspark.SparkConf = pyspark.SparkConf().set(
    "spark.driver.host", "localhost"
)
sc: SparkContext = SparkContext.getOrCreate()

# Set log level to reduce verbosity
sc.setLogLevel("WARN")

print("✅ Connected to Spark cluster!")
print(f"Spark Version: {sc.version}")
print(f"Master: {sc.master}")
print(f"App ID: {sc.applicationId}")


✅ Connected to Spark cluster!
Spark Version: 4.0.1
Master: local[*]
App ID: local-1762961465877


In [9]:
num_csv_path = "./data/processed/merged/num_2020.csv"
pre_csv_path = "./data/processed/merged/pre_2020.csv"
sub_csv_path = "./data/processed/merged/sub_2020.csv"
tag_csv_path = "./data/processed/merged/tag_2020.csv"


num_rdd = sc.textFile(num_csv_path)
pre_rdd = sc.textFile(pre_csv_path)
sub_rdd = sc.textFile(sub_csv_path)
tag_rdd = sc.textFile(tag_csv_path)

# print size of each RDD
print(f"Num RDD size: {num_rdd.count()}")
print(f"Pre RDD size: {pre_rdd.count()}")
print(f"Sub RDD size: {sub_rdd.count()}")
print(f"Tag RDD size: {tag_rdd.count()}")

Num RDD size: 11493263
Pre RDD size: 2746310
Sub RDD size: 24940
Tag RDD size: 298803


In [ ]:
import csv


# Aggregate numeric facts to inspect which tags drive the largest values each quarter.
def _parse_csv(line: str):
    return next(csv.reader([line]))


def _safe_float(value: str) -> float:
    try:
        return float(value)
    except (TypeError, ValueError):
        return 0.0


header = num_rdd.first()
parsed_num_rdd = num_rdd.filter(lambda line: line != header).map(_parse_csv)


tag_quarter_totals = parsed_num_rdd.map(
    lambda cols: ((cols[1], cols[10]), _safe_float(cols[8]))
).reduceByKey(lambda a, b: a + b)


top_tags_by_quarter = (
    tag_quarter_totals.map(lambda kv: (kv[0][1], (kv[0][0], kv[1])))
    .groupByKey()
    .mapValues(
        lambda rows: sorted(list(rows), key=lambda row: row[1], reverse=True)[
            :5
        ]
    )
    .collect()
)

for quarter, rows in sorted(top_tags_by_quarter, key=lambda item: item[0]):
    print(f"Quarter: {quarter.upper()}")
    for tag, total in rows:
        print(f"  {tag:<70} {total:,.2f}")

Here is the code to run with files stored in GCS

In [ ]:
import os

from pyspark.sql import SparkSession

# Here we use the application default credentials (ADC) for authentication
# Ensure you have run `gcloud auth application-default login` beforehand
adc_path = os.path.expanduser(
    "~/.config/gcloud/application_default_credentials.json"
)
gcs_connector_jar_path = "../gcs-connector-hadoop3-latest.jar"
if not os.path.exists(gcs_connector_jar_path):
    raise FileNotFoundError(
        f"GCS connector JAR not found at {gcs_connector_jar_path}"
    )
gcs_path = "gs://msds-694-cohort-14-3/data/num_2020.csv"

is_local = os.getenv("IS_LOCAL", "true").lower() == "true"

# Build SparkSession with conditional configuration
spark_builder = SparkSession.builder.appName("LocalGCS")

if is_local:
    spark = (
        SparkSession.builder.appName("LocalGCS")
        .config("spark.driver.host", "localhost")
        .config(
            "spark.jars",
            gcs_connector_jar_path,
        )
        .config(
            "spark.hadoop.google.cloud.auth.service.account.enable", "true"
        )
        .config(
            "spark.hadoop.google.cloud.auth.service.account.json.keyfile",
            adc_path,
        )
        .config(
            "spark.hadoop.fs.gs.impl",
            "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem",
        )
        .config(
            "spark.hadoop.fs.AbstractFileSystem.gs.impl",
            "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS",
        )
    )
else:
    spark = SparkSession.builder.appName("GCSCluster").getOrCreate()

spark = spark_builder.getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("WARN")

print("✅ Connected to Spark cluster!")
print(f"Spark Version: {sc.version}")
print(f"Master: {sc.master}")
print(f"App ID: {sc.applicationId}")

gcs_rdd = spark.sparkContext.textFile(gcs_path)
print(f"GCS RDD count: {gcs_rdd.count()}")
